# Text quality and leakage

Three checks, in the order they cost you accuracy you did not earn:

1. **Leakage** — the same text in train and valid. The score is then partly memory.
2. **Duplicates** — a corpus that is smaller than its row count.
3. **Length** — the tail that silently truncates at `max_length`.

None of these show up in a training curve. All three show up here.

In [ ]:
import orinth
import polars as pl

source = max(
    (dataset for dataset in orinth.datasets.list() if dataset.task_type == "text_classification"),
    key=lambda dataset: dataset.total_items,
)
train = orinth.datasets.load(source.id, "train", limit=20_000)
valid = orinth.datasets.load(source.id, "valid", limit=20_000)
print(source.name, "|", train.height, "train ·", valid.height, "valid")
train.head(3)

## Leakage

An exact-match check first, because it is cheap and it is what actually happens:
a dataset shuffled and split *after* duplicates were introduced. Normalising
whitespace and case before comparing catches the near-duplicates that a raw
string compare misses.

In [ ]:
def key(frame):
    return frame.select(
        pl.col("text").str.strip_chars().str.to_lowercase().str.replace_all(r"\s+", " ")
    )["text"]


shared = set(key(train).to_list()) & set(key(valid).to_list())
print(f"{len(shared)} of {valid.height} validation rows also appear in train")
if shared:
    print("\nexample:", next(iter(shared))[:200])

## Duplicates within train

Repeats are not free: they reweight the loss toward whatever was duplicated,
which is usually the boilerplate rather than the signal.

In [ ]:
duplicates = (
    train.with_columns(key(train).alias("norm"))
    .group_by("norm")
    .agg(pl.len().alias("copies"), pl.col("label").n_unique().alias("distinct_labels"))
    .filter(pl.col("copies") > 1)
    .sort("copies", descending=True)
)
print(f"{duplicates.height} repeated texts, {duplicates['copies'].sum() - duplicates.height} extra rows")

# Same text, different label. These are annotation errors, not duplicates, and
# they put a ceiling on accuracy no model can pass.
conflicting = duplicates.filter(pl.col("distinct_labels") > 1)
print(f"{conflicting.height} of them are labelled inconsistently")
conflicting.head()

## Length budget

Character length is a stand-in until you know the tokenizer; when you do, count
tokens with it instead, because the ratio is nothing like constant across
languages.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

lengths = train.select(pl.col("text").str.len_chars().alias("chars"))["chars"]
print(lengths.describe())
for quantile in (0.5, 0.9, 0.99):
    print(f"p{int(quantile * 100):<3} {lengths.quantile(quantile):>8.0f} chars")

plt.figure(figsize=(7, 2.8))
plt.hist(lengths.to_list(), bins=60)
plt.xlabel("characters")
plt.ylabel("rows")
plt.tight_layout()
plt.show()

In [ ]:
# Words are a stand-in for tokens; a subword tokenizer runs 1.2-1.5x higher on
# English and far more on anything else. Swap in the real one when you have
# picked a checkpoint — `AutoTokenizer.from_pretrained(name)` and `len(enc(text))`
# in place of the split below — and re-read this number before fixing max_length.
BUDGET = 512
words = train.select(pl.col("text").str.split(" ").list.len().alias("words"))["words"]
over = (words > BUDGET).sum()
print(f"median {words.median():.0f} words · p99 {words.quantile(0.99):.0f}")
print(f"{over}/{words.len()} rows over a {BUDGET}-token budget before subwords are counted")

## Class balance, last

Balance only means something once the rows above are real rows. A dataset that
is 40% duplicates has a class distribution describing the duplication, not the
data.

In [ ]:
train.group_by("label").agg(pl.len().alias("rows")).with_columns(
    (pl.col("rows") / train.height * 100).round(1).alias("percent")
).sort("rows", descending=True)